# FloodEye: huấn luyện trên Kaggle

Notebook chỉ làm ba việc: lấy mã ở một commit cố định, chuẩn bị dữ liệu bằng script của kho, gọi `training/train.py`. Mọi logic nằm trong kho mã.

Trước khi chạy: bật **GPU T4** và **Internet** ở khung Settings (Internet cần tài khoản đã xác minh số điện thoại).

Cách dùng:
1. Lần đầu để `MODE = "smoke"`, bấm **Run All** trong phiên tương tác: chạy 1 epoch và in thời gian, để chọn `EPOCHS` sao cho cả lần chạy dưới khoảng 10 giờ (một lần chạy nền bị cắt ở 12 giờ). Xong thì **Stop session** để không tốn hạn mức.
2. Đổi `MODE = "train"`, đặt `EPOCHS`, rồi **Save Version → Save & Run All (Commit)**. Lần chạy này tiếp tục khi đóng trình duyệt.

In [ ]:
COMMIT = "e25b81d45065e688d0725601a56069a4c7841683"  # mã commit đầy đủ; đổi khi mã huấn luyện thay đổi
CONFIG = "configs/experiments/E2_s0.yaml"
MODE = "smoke"   # "smoke": 1 epoch để đo thời gian; "train": huấn luyện thật
EPOCHS = None    # None: theo file cấu hình; đặt số để giới hạn thời gian chạy

In [ ]:
import os

if not os.path.isdir("/kaggle/working/floodeye"):
    !git clone -q https://github.com/huzy179/uav-rescue-qa.git /kaggle/working/floodeye
%cd /kaggle/working/floodeye
!git fetch -q && git checkout -q {COMMIT} && git log -1 --oneline
!pip install -q -e ".[dev]"

Các dòng `ERROR: pip's dependency resolver ...` ở ô trên bỏ qua được: chúng nói về gói Kaggle cài sẵn mà dự án không dùng (`bigframes`, `google-colab`, `cudf`, ...). Ô dưới kiểm tra những gói dự án thật sự cần.

In [ ]:
import torch, cv2, ultralytics

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| số GPU", torch.cuda.device_count())
print("opencv", cv2.__version__, "| ultralytics", ultralytics.__version__)
assert torch.cuda.is_available(), "Chưa có GPU: bật Accelerator GPU T4 ở khung Settings"

## Dữ liệu

VisDrone train và val, chuyển về một lớp `person`. Đã tải rồi thì bỏ qua bước tải.

In [ ]:
%%bash
set -e
mkdir -p data/raw/visdrone
for split in train val; do
  if [ ! -d data/raw/visdrone/VisDrone2019-DET-$split ]; then
    (cd data/raw/visdrone && curl -sLO https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-$split.zip \
      && unzip -q VisDrone2019-DET-$split.zip && rm VisDrone2019-DET-$split.zip)
  fi
  python tools/convert_visdrone.py --src data/raw/visdrone/VisDrone2019-DET-$split --split $split
done

## Huấn luyện

`smoke` ghi vào `runs/train/smoke` để không lẫn với kết quả thật. Phiên bị ngắt thì chạy lại ô dưới với `--resume` thêm vào cuối lệnh, sau khi đã chép `runs/train/` của phiên trước về đúng chỗ.

In [ ]:
import time

if MODE == "smoke":
    args = "--epochs 1 --output runs/train/smoke"
elif MODE == "train":
    args = f"--epochs {EPOCHS}" if EPOCHS else ""
else:
    raise ValueError(f"MODE phải là 'smoke' hoặc 'train', nhận được {MODE!r}")

start = time.perf_counter()
!python training/train.py --config {CONFIG} --device 0 {args}
print(f"Thời gian chạy ({MODE}): {(time.perf_counter() - start) / 60:.1f} phút")

## Dọn dẹp

Xoá dữ liệu để phần Output của notebook chỉ còn mã và `runs/train/` (trọng số, `results.csv`, `run_config.json`). Lưu Output thành Kaggle Dataset có phiên bản rồi tải về `runs/train/` trên máy.

In [ ]:
!rm -rf data